In [ ]:
import pandas as pd
import numpy as np
np.seterr(divide='ignore', over='ignore', invalid='ignore')

import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dropout, Dense, Flatten, BatchNormalization

from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve, precision_recall_curve, auc
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

import pickle
import time, sys
sys.path.insert(0, 'src')

from lime import lime_tabular

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.loader import NeighborLoader
from torch_geometric.data import Data

years = np.arange(2009, 2024)

In [ ]:
spese = {}
entrate = {}
anticipazioni = {}
indicatori = {}

for year in years:
    spese[f'{year}'] = pd.read_csv(f'data/Spese/Spese {year}.csv', index_col='BDAP')
    entrate[f'{year}'] = pd.read_csv(f'data/Entrate/Entrate {year}.csv', index_col='BDAP')
    anticipazioni[f'{year}'] = pd.read_csv(f'data/Anticipazioni/Anticipazioni {year}.csv', index_col='BDAP')
    if year >= 2016:
        indicatori[f'{year}'] = pd.read_csv(f'data/Indicatori/Indicatori {year}.csv', index_col='BDAP')
zone = pd.read_csv('data/zona.csv', index_col='BDAP')
pop = pd.read_csv('data/popolazione.csv', index_col='BDAP')
comuni = pd.read_csv('data/comuni.csv', index_col='BDAP')

critici = pd.read_csv('data//CriticitàComuni.csv', sep = ';', encoding = 'latin_1', low_memory = False)
for year in np.arange(2000, 2025):
    comuni[f'Target {year}'] = 0
for year in np.arange(2000, 2025):
    for idx, row in critici.iterrows():
        if int(row['Anno']) == year:
            comune = row['Comune']
            comune_2 = comune.replace('-', ' ')
            comuni.loc[(comuni['Comune'] == f'COMUNE DI {comune}') | (comuni['Comune'] == f'COMUNE DI {comune_2}'), f'Target {year}'] = 1

comuni_ind = comuni.loc[indicatori['2016'].index]



In [ ]:
columns = [f'Target {year}' for year in np.arange(2009, 2024)]

In [ ]:
comuni[columns].sum()

In [ ]:
comuni['Regione'].unique()

In [ ]:
RSS = ["VALLE D'AOSTA/VALLÉE D'AOSTE", 'TRENTINO-ALTO ADIGE/SÜDTIROL', 'FRIULI VENEZIA GIULIA', 'SARDEGNA']
comuni_rss = comuni[~comuni['Regione'].isin(RSS)]
comuni_ind_rss = comuni_ind[~comuni_ind['Regione'].isin(RSS)]

# Functions

In [ ]:
from bankit import (
    create_data,
    evaluate_model,
    filter_distances,
    haversine_distance,
    inverse_distance,
    run_training,
    top_distances,
)


# EXP

In [ ]:
comuni = comuni_rss
comuni_ind = comuni_ind_rss

In [ ]:
def run_dataset_experiment(name, periodo, comuni_d, pop_d, spese_d, entrate_d, zone_d, anticipazioni_d, indicatori_d, epochs, batch_size, save):
    reshape = False if zone_d is None else True

    results = {}
    models = {}

    for target in [1, 2, 3]:
        model = {}
        result = {}
        for input in [4, 5, 6]:
            print(f'Target depth: {target}\tInput depth: {input}')

            X, y = create_data(comuni=comuni_d, pop=pop_d, spese=spese_d, entrate=entrate_d, zone=zone_d,
                                anticipazioni=anticipazioni_d, indicatori=indicatori_d, input=input, target=target, periodo=periodo)
            model[input], result[input] = run_training(X, y, comuni_d, epochs=epochs, batch_size=batch_size, reshape=reshape)

        results[target] = result
        models[target] = model

    if save:
        with open(f'results_{name}_periodo{periodo}.pickle', 'wb') as handle:
            pickle.dump(results, handle, protocol=pickle.HIGHEST_PROTOCOL)
        with open(f'models_{name}_periodo{periodo}.pickle', 'wb') as handle:
            pickle.dump(models, handle, protocol=pickle.HIGHEST_PROTOCOL)

    return results, models


## Periodo 1

In [ ]:
spese_impegno = {}
for key, elem in spese.items():
    spese_impegno[key] = elem[[col for col in elem.columns if 'Impegno' in col or 'Impegni' in col]]

entrate_acc = {}
for key, elem in entrate.items():
    entrate_acc[key] = elem[[col for col in elem.columns if 'Accertamento' in col or 'Accertamenti' in col]]


In [ ]:
save = 1
epochs = 400
batch_size = 512
periodo = 1

periodo1_configs = [
    ('BDAP', comuni, pop, spese, entrate, zone, None, None),
    ('BDAP_anticipazioni', comuni, pop, spese, entrate, zone, anticipazioni, None),
    ('anticipazioni', comuni, pop, None, None, zone, anticipazioni, None),
    ('BDAP_anticipazioni_new', comuni, pop, spese_impegno, entrate_acc, zone, anticipazioni, None),
]

results = {}
models = {}
for name, comuni_d, pop_d, spese_d, entrate_d, zone_d, anticipazioni_d, indicatori_d in periodo1_configs:
    print(f'--- {name} (periodo {periodo}) ---')
    results[name], models[name] = run_dataset_experiment(
        name, periodo, comuni_d, pop_d, spese_d, entrate_d, zone_d, anticipazioni_d, indicatori_d,
        epochs=epochs, batch_size=batch_size, save=save)


## Periodo 2

In [ ]:
save = 1
epochs = 400
batch_size = 512
periodo = 2

periodo2_configs = [
    ('BDAP', comuni, pop, spese, entrate, zone, None, None),
    ('BDAP_anticipazioni', comuni, pop, spese, entrate, zone, anticipazioni, None),
    ('anticipazioni', comuni, pop, None, None, zone, anticipazioni, None),
    ('indicatori', comuni_ind, None, None, None, None, None, indicatori),
    ('indicatori_anticipazioni', comuni_ind, None, None, None, None, anticipazioni, indicatori),
    ('BDAP_indicatori_anticipazioni', comuni_ind, pop, spese, entrate, zone, anticipazioni, indicatori),
    ('BDAP_anticipazioni_new', comuni, pop, spese_impegno, entrate_acc, zone, anticipazioni, None),
]

results = {}
models = {}
for name, comuni_d, pop_d, spese_d, entrate_d, zone_d, anticipazioni_d, indicatori_d in periodo2_configs:
    print(f'--- {name} (periodo {periodo}) ---')
    results[name], models[name] = run_dataset_experiment(
        name, periodo, comuni_d, pop_d, spese_d, entrate_d, zone_d, anticipazioni_d, indicatori_d,
        epochs=epochs, batch_size=batch_size, save=save)


# LIME

## EXP

In [ ]:
def cnn_pred(sample):
    p2 = model.predict(sample, verbose = 0)
    p1 = 1.0 - p2
    return np.array([p1, p2]).T[0]

def compact_list(sample, features_names):
    output = []
    for elem1 in features_names:
        t = 0
        for elem2 in sample:
            if elem2[0][:-4] == elem1:
                t += elem2[1]
        output.append((elem1,t))
    return output

In [ ]:
results_files = 'results/models_BDAP_indicatori_anticipazioni_periodo2.pickle'
model_files = 'results//models_BDAP_indicatori_anticipazioni_periodo2.pickle'

s = len(spese['2023'].columns) // 3
e = len(entrate['2023'].columns) // 3

features_names = [f'S{i+1} IMPEGNI' for i in range(s)] + [f'S{i+1} PAGAMENTI' for i in range(s)] + [f'S{i+1} RESIDUI' for i in range(s)]
features_names += [f'E{i+1} ACCERTAMENTI' for i in range(e)] + [f'E{i+1} RISCOSSIONI' for i in range(e)] + [f'E{i+1} RESIDUI' for i in range(e)]
features_names += [f'I{i+1}' for i in range(10)]
features_names += [f'A{i+1}' for i in range(5)]
features_names += ['P'] + [f'Z{i+1}' for i in range(5)]

with open(results_files, 'rb') as file:
        results = pickle.load(file)
with open(model_files, 'rb') as file:
        models = pickle.load(file)

In [ ]:
periodo = 2
comuni_d = comuni_ind
pop_d = pop
spese_d = spese
entrate_d = entrate
zone_d = zone
anticipazioni_d = anticipazioni
indicatori_d = indicatori

t = []

for target, target_dict in models.items():
    for input, input_dict in target_dict.items():
        
        model = input_dict['CNN']

        X, y = create_data(comuni = comuni_d, pop = pop_d, spese = spese_d, entrate = entrate_d, zone = zone_d, anticipazioni = anticipazioni_d, indicatori = indicatori_d, input = input, target = target, periodo=periodo)
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

        explainer = lime_tabular.RecurrentTabularExplainer(X_train ,training_labels=y_train, feature_names=features_names, discretize_continuous=False, feature_selection='auto', class_names=['Negative','Positive'])
        if len(y_test) > 1000:
            for i in np.random.choice(range(len(y_test)), size=1000, replace=False):
                exp = explainer.explain_instance((X_test[i]), cnn_pred, num_features=1000)
                t.append(compact_list(exp.as_list(), features_names))
        else:
            for i in range(len(y_test)):
                exp = explainer.explain_instance((X_test[i]), cnn_pred, num_features=1000)
                t.append(compact_list(exp.as_list(), features_names))

        print(f'Completed explanations for target {target} input {input}')

with open('lime_BDAP_indicatori_anticipazioni_periodo2.pickle', 'wb') as handle:
    pickle.dump(t, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
results_files = 'results/models_BDAP_periodo1.pickle'
model_files = 'results//models_BDAP_periodo1.pickle'

s = len(spese['2009'].columns) // 3
e = len(entrate['2009'].columns) // 3

features_names = [f'S{i+1} IMPEGNI' for i in range(s)] + [f'S{i+1} PAGAMENTI' for i in range(s)] + [f'S{i+1} RESIDUI' for i in range(s)]
features_names += [f'E{i+1} ACCERTAMENTI' for i in range(e)] + [f'E{i+1} RISCOSSIONI' for i in range(e)] + [f'E{i+1} RESIDUI' for i in range(e)]
features_names += ['P'] + [f'Z{i+1}' for i in range(5)]

with open(results_files, 'rb') as file:
        results = pickle.load(file)
with open(model_files, 'rb') as file:
        models = pickle.load(file)

In [ ]:
periodo = 1
comuni_d = comuni
pop_d = pop
spese_d = spese
entrate_d = entrate
zone_d = zone
anticipazioni_d = None
indicatori_d = None

t = []

for target, target_dict in models.items():
    for input, input_dict in target_dict.items():
        
        model = input_dict['CNN']

        X, y = create_data(comuni = comuni_d, pop = pop_d, spese = spese_d, entrate = entrate_d, zone = zone_d, anticipazioni = anticipazioni_d, indicatori = indicatori_d, input = input, target = target, periodo=periodo)
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

        explainer = lime_tabular.RecurrentTabularExplainer(X_train ,training_labels=y_train, feature_names=features_names, discretize_continuous=False, feature_selection='auto', class_names=['Negative','Positive'])
        if len(y_test) > 1000:
            for i in np.random.choice(range(len(y_test)), size=1000, replace=False):
                exp = explainer.explain_instance((X_test[i]), cnn_pred, num_features=1000)
                t.append(compact_list(exp.as_list(), features_names))
        else:
            for i in range(len(y_test)):
                exp = explainer.explain_instance((X_test[i]), cnn_pred, num_features=1000)
                t.append(compact_list(exp.as_list(), features_names))
                
        print(f'Completed explanations for target {target} input {input}')

with open('lime_BDAP_periodo1.pickle', 'wb') as handle:
    pickle.dump(t, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
results_files = 'results/models_BDAP_periodo2.pickle'
model_files = 'results//models_BDAP_periodo2.pickle'

s = len(spese['2023'].columns) // 3
e = len(entrate['2023'].columns) // 3

features_names = [f'S{i+1} IMPEGNI' for i in range(s)] + [f'S{i+1} PAGAMENTI' for i in range(s)] + [f'S{i+1} RESIDUI' for i in range(s)]
features_names += [f'E{i+1} ACCERTAMENTI' for i in range(e)] + [f'E{i+1} RISCOSSIONI' for i in range(e)] + [f'E{i+1} RESIDUI' for i in range(e)]
features_names += ['P'] + [f'Z{i+1}' for i in range(5)]

with open(results_files, 'rb') as file:
        results = pickle.load(file)
with open(model_files, 'rb') as file:
        models = pickle.load(file)

In [ ]:
periodo = 2
comuni_d = comuni
pop_d = pop
spese_d = spese
entrate_d = entrate
zone_d = zone
anticipazioni_d = None
indicatori_d = None

t = []

for target, target_dict in models.items():
    for input, input_dict in target_dict.items():
        
        model = input_dict['CNN']

        X, y = create_data(comuni = comuni_d, pop = pop_d, spese = spese_d, entrate = entrate_d, zone = zone_d, anticipazioni = anticipazioni_d, indicatori = indicatori_d, input = input, target = target, periodo=periodo)
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

        explainer = lime_tabular.RecurrentTabularExplainer(X_train ,training_labels=y_train, feature_names=features_names, discretize_continuous=False, feature_selection='auto', class_names=['Negative','Positive'])
        if len(y_test) > 1000:
            for i in np.random.choice(range(len(y_test)), size=1000, replace=False):
                exp = explainer.explain_instance((X_test[i]), cnn_pred, num_features=1000)
                t.append(compact_list(exp.as_list(), features_names))
        else:
            for i in range(len(y_test)):
                exp = explainer.explain_instance((X_test[i]), cnn_pred, num_features=1000)
                t.append(compact_list(exp.as_list(), features_names))
                
        print(f'Completed explanations for target {target} input {input}')

with open('lime_BDAP_periodo2.pickle', 'wb') as handle:
    pickle.dump(t, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
results_files = 'results/models_indicatori_periodo2.pickle'
model_files = 'results//models_indicatori_periodo2.pickle'

features_names = [f'I{i+1}' for i in range(10)]

with open(results_files, 'rb') as file:
        results = pickle.load(file)
with open(model_files, 'rb') as file:
        models = pickle.load(file)

In [ ]:
periodo = 2
comuni_d = comuni_ind
pop_d = None
spese_d = None
entrate_d = None
zone_d = None
anticipazioni_d = None
indicatori_d = indicatori

t = []

for target, target_dict in models.items():
    for input, input_dict in target_dict.items():
        
        model = input_dict['CNN']

        X, y = create_data(comuni = comuni_d, pop = pop_d, spese = spese_d, entrate = entrate_d, zone = zone_d, anticipazioni = anticipazioni_d, indicatori = indicatori_d, input = input, target = target, periodo=periodo)
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

        explainer = lime_tabular.RecurrentTabularExplainer(X_train ,training_labels=y_train, feature_names=features_names, discretize_continuous=False, feature_selection='auto', class_names=['Negative','Positive'])
        if len(y_test) > 1000:
            for i in np.random.choice(range(len(y_test)), size=1000, replace=False):
                exp = explainer.explain_instance((X_test[i]), cnn_pred, num_features=1000)
                t.append(compact_list(exp.as_list(), features_names))
        else:
            for i in range(len(y_test)):
                exp = explainer.explain_instance((X_test[i]), cnn_pred, num_features=1000)
                t.append(compact_list(exp.as_list(), features_names))
                
        print(f'Completed explanations for target {target} input {input}')

with open('lime_indicatori_periodo2.pickle', 'wb') as handle:
    pickle.dump(t, handle, protocol=pickle.HIGHEST_PROTOCOL)

## Plot

In [ ]:
with open('results/lime_BDAP_indicatori_anticipazioni_periodo2.pickle', 'rb') as file:
        lime_tot = pickle.load(file)

with open('results/lime_BDAP_periodo1.pickle', 'rb') as file:
        lime_p1 = pickle.load(file)

with open('results/lime_BDAP_periodo2.pickle', 'rb') as file:
        lime_p2 = pickle.load(file)

with open('results/lime_indicatori_periodo2.pickle', 'rb') as file:
        lime_ind = pickle.load(file)


In [ ]:
def with_sub(label: str, sub: str | None = None) -> str:
    return rf"${label}_{{{sub}}}$" if sub is not None else label

In [ ]:
sub = "P1"

replacements = {}

for i in range(1, 10):
    replacements[f"S{i} RESIDUI"]      = f"{with_sub(f'S{i}', sub)} PAY PY"
    replacements[f"E{i} RESIDUI"]      = f"{with_sub(f'E{i}', sub)} COL PY"
    replacements[f"S{i} IMPEGNI"]      = f"{with_sub(f'S{i}', sub)} COM"
    replacements[f"S{i} PAGAMENTI"]    = f"{with_sub(f'S{i}', sub)} PAY CY"
    replacements[f"E{i} ACCERTAMENTI"] = f"{with_sub(f'E{i}', sub)} REC"
    replacements[f"E{i} RISCOSSIONI"]  = f"{with_sub(f'E{i}', sub)} COL CY"

feats_p1 = [elem[0] for elem in lime_p1[0]]
feats_p1 = [replacements.get(feat, feat) for feat in feats_p1]
lp1 = {}
for feat in feats_p1:
    lp1[feat] = []
for elem in lime_p1:
    for feat, elem2 in zip(feats_p1, elem):
        lp1[feat].append(elem2[1])

In [ ]:
sub = "P2"

replacements = {}

for i in range(1, 10):
    replacements[f"S{i} RESIDUI"]      = f"{with_sub(f'S{i}', sub)} PAY PY"
    replacements[f"E{i} RESIDUI"]      = f"{with_sub(f'E{i}', sub)} COL PY"
    replacements[f"S{i} IMPEGNI"]      = f"{with_sub(f'S{i}', sub)} COM"
    replacements[f"S{i} PAGAMENTI"]    = f"{with_sub(f'S{i}', sub)} PAY CY"
    replacements[f"E{i} ACCERTAMENTI"] = f"{with_sub(f'E{i}', sub)} REC"
    replacements[f"E{i} RISCOSSIONI"]  = f"{with_sub(f'E{i}', sub)} COL CY"

feats_tot = [elem[0] for elem in lime_tot[0]]
feats_tot = [replacements.get(feat, feat) for feat in feats_tot]
lt = {}
for feat in feats_tot:
    lt[feat] = []
for elem in lime_tot:
    for feat, elem2 in zip(feats_tot, elem):
        lt[feat].append(elem2[1])

feats_p2 = [elem[0] for elem in lime_p2[0]]
feats_p2 = [replacements.get(feat, feat) for feat in feats_p2]
lp2 = {}
for feat in feats_p2:
    lp2[feat] = []
for elem in lime_p2:
    for feat, elem2 in zip(feats_p2, elem):
        lp2[feat].append(elem2[1])

feats_ind = [elem[0] for elem in lime_ind[0]]
lind = {}
for feat in feats_ind:
    lind[feat] = []
for elem in lime_ind:
    for feat, elem2 in zip(feats_ind, elem):
        lind[feat].append(elem2[1])


In [ ]:
def plot_lime_boxplot(values, tick_labels, filename, save):
    plt.boxplot(values, whis=[5, 95], vert=False, tick_labels=tick_labels, showfliers=False)
    plt.axvline(x=0, color='red', linestyle='--', linewidth=1)
    plt.xlabel('Lime score')
    plt.tight_layout()
    if save:
        plt.savefig(filename, bbox_inches='tight')
    plt.show()


def plot_lime_extreme(scores_dict, feats, k, bottom, filename, save):
    values = [scores_dict[f] for f in feats]
    avg = np.array([np.mean(v) for v in values])
    idx = [int(i) for i in (np.argsort(avg)[:k] if bottom else np.argsort(avg)[-k:])]
    plot_lime_boxplot([values[i] for i in idx], [feats[i] for i in idx], filename, save)


def plot_lime_errorbar(scores_dict, feats, filename, save):
    values = [scores_dict[f] for f in feats]
    avg = np.array([np.mean(v) for v in values])
    p5 = np.array([np.percentile(v, 5) for v in values])
    p95 = np.array([np.percentile(v, 95) for v in values])

    sorted_idx = np.argsort(avg)
    sorted_avg = avg[sorted_idx]
    lower_err = sorted_avg - p5[sorted_idx]
    upper_err = p95[sorted_idx] - sorted_avg

    plt.errorbar(sorted_avg, np.arange(len(avg)), xerr=[lower_err, upper_err], fmt='o',
                 ecolor='grey', elinewidth=0.8, capsize=0, markersize=2, color='black')
    plt.axvline(x=0, color='red', linestyle='--', linewidth=1)
    plt.yticks([])
    plt.xlabel('Lime score')
    plt.tight_layout()
    if save:
        plt.savefig(filename, bbox_inches='tight')
    plt.show()


In [ ]:
save = 1
k = 10
plot_lime_extreme(lt, feats_tot, k, bottom=True, filename=f'lime_BDAP_indicatori_anticipazioni_periodo2_BOT{k}.png', save=save)


In [ ]:
save = 1
k = 10
plot_lime_extreme(lt, feats_tot, k, bottom=False, filename=f'lime_BDAP_indicatori_anticipazioni_periodo2_TOP{k}.png', save=save)


In [ ]:
save = 1
plot_lime_errorbar(lt, feats_tot, 'lime_BDAP_indicatori_anticipazioni_periodo2_TOT.png', save)


In [ ]:
save = 1
k = 10
plot_lime_extreme(lp1, feats_p1, k, bottom=True, filename=f'lime_BDAP_periodo1_BOT{k}.png', save=save)


In [ ]:
save = 1
k = 10
plot_lime_extreme(lp1, feats_p1, k, bottom=False, filename=f'lime_BDAP_periodo1_TOP{k}.png', save=save)


In [ ]:
save = 1
plot_lime_errorbar(lp1, feats_p1, 'lime_BDAP_periodo1_TOT.png', save)


In [ ]:
save = 1
k = 10
plot_lime_extreme(lp2, feats_p2, k, bottom=True, filename=f'lime_BDAP_periodo2_BOT{k}.png', save=save)


In [ ]:
save = 1
k = 10
plot_lime_extreme(lp2, feats_p2, k, bottom=False, filename=f'lime_BDAP_periodo2_TOP{k}.png', save=save)


In [ ]:
save = 1
plot_lime_errorbar(lp2, feats_p2, 'lime_BDAP_periodo2_TOT.png', save)


In [ ]:
save = 1
plot_lime_boxplot([lind[f] for f in feats_ind], feats_ind, 'lime_indicatori_periodo2.png', save)


# MDI

In [ ]:
def sum_scores(scores, input, zone):
    if zone:
        temp = scores[:-5]
        temp = temp.reshape((input, -1))
        summed = np.sum(temp, axis=0)
        summed = np.concatenate((summed, scores[-5:]*input))
    else:
        temp = scores.reshape((input, -1))
        summed = np.sum(temp, axis=0)
    return summed

In [ ]:
def with_sub(prefix: str, i: int, sub: str | None = None) -> str:
    base = f"{prefix}{i}"
    return rf"${base}_{{{sub}}}$" if sub is not None else base

In [ ]:
def plot_mdi_top(models_dict, features_names, zone, filename, save, top_k=None):
    t = np.zeros((len(features_names),))
    i = 0
    for target, target_dict in models_dict.items():
        for input, input_dict in target_dict.items():
            model = input_dict['Random Forest']
            scores = sum_scores(model.feature_importances_, input, zone=zone)
            t += scores
            i += 1
    t = t / i

    if top_k is not None:
        idx = np.argsort(t)[-top_k:]
        labels = [features_names[j] for j in idx]
        values = t[idx]
    else:
        labels = features_names
        values = t

    plt.barh(labels, values)
    if save:
        plt.savefig(filename, bbox_inches='tight')
    plt.show()


def plot_mdi_errorbar(models_dict, features_names, zone, filename, save):
    t = []
    for target, target_dict in models_dict.items():
        for input, input_dict in target_dict.items():
            model = input_dict['Random Forest']
            scores = sum_scores(model.feature_importances_, input, zone=zone)
            t.append(list(scores))

    values = [[elem[j] for elem in t] for j in range(len(features_names))]
    avg = np.array([np.mean(v) for v in values])
    p5 = np.array([np.percentile(v, 5) for v in values])
    p95 = np.array([np.percentile(v, 95) for v in values])

    sorted_idx = np.argsort(avg)
    sorted_avg = avg[sorted_idx]
    lower_err = sorted_avg - p5[sorted_idx]
    upper_err = p95[sorted_idx] - sorted_avg

    plt.errorbar(sorted_avg, np.arange(len(avg)), xerr=[lower_err, upper_err], fmt='o',
                 ecolor='grey', elinewidth=0.8, capsize=0, markersize=2, color='black')
    plt.axvline(x=0, color='red', linestyle='--', linewidth=1)
    plt.yticks([])
    plt.xlabel('MDI score')
    plt.tight_layout()
    if save:
        plt.savefig(filename, bbox_inches='tight')
    plt.show()


In [ ]:
results_files = 'results/models_BDAP_indicatori_anticipazioni_periodo2.pickle'
model_files = 'results//models_BDAP_indicatori_anticipazioni_periodo2.pickle'

s = len(spese['2023'].columns) // 3
e = len(entrate['2023'].columns) // 3

sub = "P2"

features_names = (
    [f"{with_sub('S', i+1, sub)} COM"    for i in range(s)]
    + [f"{with_sub('S', i+1, sub)} PAY CY" for i in range(s)]
    + [f"{with_sub('S', i+1, sub)} PAY PY" for i in range(s)]
    + [f"{with_sub('E', i+1, sub)} REC" for i in range(e)]
    + [f"{with_sub('E', i+1, sub)} COL CY" for i in range(e)]
    + [f"{with_sub('E', i+1, sub)} COL PY" for i in range(e)]
)

features_names += [f'I{i+1}' for i in range(10)]
features_names += [f'A{i+1}' for i in range(5)]
features_names += ['P'] + [f'Z{i+1}' for i in range(5)]

with open(results_files, 'rb') as file:
        results = pickle.load(file)
with open(model_files, 'rb') as file:
        models = pickle.load(file)

In [ ]:
save = 1
plot_mdi_top(models, features_names, zone=True, filename='mdi_BDAP_indicatori_anticipazioni_periodo2.png', save=save)


In [ ]:
results_files = 'results/models_BDAP_periodo1.pickle'
model_files = 'results//models_BDAP_periodo1.pickle'

s = len(spese['2009'].columns) // 3
e = len(entrate['2009'].columns) // 3

sub = "P1"

features_names = (
    [f"{with_sub('S', i+1, sub)} COM"    for i in range(s)]
    + [f"{with_sub('S', i+1, sub)} PAY CY" for i in range(s)]
    + [f"{with_sub('S', i+1, sub)} PAY PY" for i in range(s)]
    + [f"{with_sub('E', i+1, sub)} REC" for i in range(e)]
    + [f"{with_sub('E', i+1, sub)} COL CY" for i in range(e)]
    + [f"{with_sub('E', i+1, sub)} COL PY" for i in range(e)]
)
features_names += ['P'] + [f'Z{i+1}' for i in range(5)]

with open(results_files, 'rb') as file:
        results = pickle.load(file)
with open(model_files, 'rb') as file:
        models = pickle.load(file)

In [ ]:
save = 1
plot_mdi_top(models, features_names, zone=True, filename='mdi_BDAP_periodo1.png', save=save)


In [ ]:
save = 1
plot_mdi_errorbar(models, features_names, zone=True, filename='MDI_BDAP_periodo1_TOT.png', save=save)


In [ ]:
results_files = 'results/models_BDAP_periodo2.pickle'
model_files = 'results//models_BDAP_periodo2.pickle'

s = len(spese['2023'].columns) // 3
e = len(entrate['2023'].columns) // 3

sub = "P2"

features_names = (
    [f"{with_sub('S', i+1, sub)} COM"    for i in range(s)]
    + [f"{with_sub('S', i+1, sub)} PAY CY" for i in range(s)]
    + [f"{with_sub('S', i+1, sub)} PAY PY" for i in range(s)]
    + [f"{with_sub('E', i+1, sub)} REC" for i in range(e)]
    + [f"{with_sub('E', i+1, sub)} COL CY" for i in range(e)]
    + [f"{with_sub('E', i+1, sub)} COL PY" for i in range(e)]
)
features_names += ['P'] + [f'Z{i+1}' for i in range(5)]

with open(results_files, 'rb') as file:
        results = pickle.load(file)
with open(model_files, 'rb') as file:
        models = pickle.load(file)

In [ ]:
save = 1
plot_mdi_top(models, features_names, zone=True, filename='mdi_BDAP_periodo2.png', save=save)


In [ ]:
save = 1
plot_mdi_errorbar(models, features_names, zone=True, filename='MDI_BDAP_periodo2_TOT.png', save=save)


In [ ]:
results_files = 'results/models_indicatori_periodo2.pickle'
model_files = 'results//models_indicatori_periodo2.pickle'

features_names = [f'I{i+1}' for i in range(10)]

with open(results_files, 'rb') as file:
        results = pickle.load(file)
with open(model_files, 'rb') as file:
        models = pickle.load(file)

In [ ]:
save = 1
plot_mdi_top(models, features_names, zone=False, filename='mdi_indicatori_periodo2.png', save=save)


# Graphs

## Load

In [ ]:
columns = ['Denominazione', 'Dizione_Provincia', 'Dizione_Regione', 'Dizione_zona', 'Codice_ISTAT_Comune']

final = pd.read_csv('data//Anagrafe_comuni.csv', sep = ';', encoding = 'latin_1', low_memory = False).set_index('Id_Ente')

final['Data_Istituzione'] = pd.to_datetime(final['Data_Istituzione'], format='%Y-%m-%d')
final = final[final['Data_Istituzione'].dt.year < 2008]

final = final[final['Data_Cessazione'].isna()]

final = final[final['Codice_Tipologia_DLGS_118_2011'] == 'ELCOMU'][columns]

final = final.rename(columns={'Denominazione': 'Comune', 'Dizione_Provincia': 'Provincia', 'Dizione_Regione': 'Regione', 'Dizione_zona': 'Zona'})
final.index.names = ['BDAP']

final.loc[602642930517270801, "Codice_ISTAT_Comune"] = 21014

final

In [ ]:
gi_comuni_df = pd.read_csv('gi_comuni.csv', sep = ';', decimal=',', encoding = 'latin_1', low_memory = False)
gi_comuni_df['lat'] = gi_comuni_df['lat'].astype(float)
gi_comuni_df['lon'] = gi_comuni_df['lon'].astype(float)
gi_comuni_df.head()

In [ ]:
temp = []
for row in gi_comuni_df.iterrows():
    codice_istat = row[1]['codice_istat']
    matching_indices = final[final['Codice_ISTAT_Comune'] == codice_istat].index
    if matching_indices.size > 1:
        print(f"Multiple matches found for ISTAT code {codice_istat}: {matching_indices.tolist()}")
    if not matching_indices.empty:
        temp.append(matching_indices[-1])
    else:
        temp.append(np.nan)
gi_comuni_df['BDAP'] = temp
gi_comuni_df = gi_comuni_df.dropna(subset=['BDAP'])
gi_comuni_df = gi_comuni_df.set_index('BDAP')
gi_comuni_df.sort_index(inplace=True)
gi_comuni_df.index = gi_comuni_df.index.astype(int)

In [ ]:
distance_matrix = np.zeros((len(gi_comuni_df), len(gi_comuni_df)))
i = 0
for row in gi_comuni_df.itertuples():
    lat1, lon1 = row.lat, row.lon
    distances = haversine_distance(lat1, lon1, gi_comuni_df['lat'], gi_comuni_df['lon'])
    distance_matrix[i, :] = distances
    i += 1

## Graphs

In [ ]:
for top_k in [1, 3, 5, 10, 20, 50]:
    temp = top_distances(distance_matrix, top_k)
    inv_temp = inverse_distance(temp, power=1)
    n = (inv_temp != 0).sum().sum()
    ratio = n / (inv_temp.shape[0] * inv_temp.shape[1])
    print(f"Top {top_k} Non-zero entries: {n}, Ratio: {ratio:.4f}")

In [ ]:
d = [(0,0,0)]
for distance in range(1,102):
    temp = filter_distances(distance_matrix, distance)
    inv_temp = inverse_distance(temp, power=1)
    n = (inv_temp != 0).sum().sum()
    ratio = n / (inv_temp.shape[0] * (inv_temp.shape[1]-1))
    print(f'Distance: {distance} km, Non-zero entries: {n}, Ratio: {ratio:.4f}')
    d.append((distance, n, ratio))

In [ ]:
save = 0

distances = np.array([t[0] for t in d])
ratios = np.array([t[2] for t in d])

plt.figure(figsize=(8,5))
plt.plot(distances, 100*ratios, linestyle='-')
plt.xlabel('Distance (km)')
plt.ylabel('Percentage of non-zero entries')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlim(0,100)
plt.ylim(0, max(ratios)*1.1)
plt.xticks(np.arange(0, 101, 5))
plt.yticks(np.arange(0, max(100*ratios)*1.1, 1))
plt.tight_layout()
if save:
    plt.savefig('distance_ratio.png')
plt.show()

# Falsi Positivi

In [ ]:
models_files = 'results//models_BDAP_periodo1.pickle'
models = {}
with open(models_files, 'rb') as file:
        models = pickle.load(file)


In [ ]:
t = ('BDAP', 6, 3, 'Random Forest')
model = models[3][6]['Logistic Regression']

In [ ]:
input = 6
periodo = 1
target = 3


comuni_d = comuni
pop_d = pop
spese_d = spese
entrate_d = entrate
zone_d = zone
anticipazioni_d = None
indicatori_d = None
X, y = create_data(comuni = comuni_d, pop = pop_d, spese = spese_d, entrate = entrate_d, zone = zone_d, anticipazioni = anticipazioni_d, indicatori = indicatori_d, input = input, target = target, periodo=periodo)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

X_train_reshaped = np.concatenate([X_train[:, :, :-5].reshape(X_train.shape[0], -1), X_train[:, 0, -5:]], axis=-1)
X_test_reshaped = np.concatenate([X_test[:, :, :-5].reshape(X_test.shape[0], -1), X_test[:, 0, -5:]], axis=-1)

In [ ]:
X_test_reshaped.shape
y_test.shape

In [ ]:
res, pred_test, pred_train = evaluate_model(model, X_train_reshaped,y_train, X_test_reshaped, y_test)

# Parametri Obbiettivi

In [ ]:
parametri = pd.read_excel("data/parametri_obiettivi.xlsx")

In [ ]:
columns = ['Denominazione', 'Dizione_Provincia', 'Dizione_Regione', 'Dizione_zona']

final = pd.read_csv('data//Anagrafe_comuni.csv', sep = ';', encoding = 'latin_1', low_memory = False).set_index('Id_Ente')

final['Data_Istituzione'] = pd.to_datetime(final['Data_Istituzione'], format='%Y-%m-%d')
final = final[final['Data_Istituzione'].dt.year < 2009]

final['Data_Cessazione'] = pd.to_datetime(
    final['Data_Cessazione'],
    format='%Y-%m-%d',
    errors='coerce'

).fillna(pd.Timestamp('2262-01-01'))

final = final[
    (final['Data_Istituzione'].dt.year < 2009)
    & (final['Data_Cessazione'].dt.year >= 2024)
]

final = final[final['Descr_Tipologia_SIOPE'] == 'COMUNI'][columns]

final = final.rename(columns={'Denominazione': 'Comune', 'Dizione_Provincia': 'Provincia', 'Dizione_Regione': 'Regione', 'Dizione_zona': 'Zona'})
final.index.names = ['BDAP']

final
comuni = final

critici = pd.read_csv('data//CriticitàComuni.csv', sep = ';', encoding = 'latin_1', low_memory = False)
for year in np.arange(2000, 2025):
    comuni[f'Target {year}'] = 0
for year in np.arange(2000, 2025):
    for idx, row in critici.iterrows():
        if int(row['Anno']) == year:
            comune = row['Comune']
            comune_2 = comune.replace('-', ' ')
            comuni.loc[(comuni['Comune'] == f'COMUNE DI {comune}') | (comuni['Comune'] == f'COMUNE DI {comune_2}'), f'Target {year}'] = 1


In [ ]:
parametri_target = pd.DataFrame(index = comuni.index)
for year in np.arange(2009, 2016):
    exact_matches = []
    temp = parametri.loc[parametri['eserciziofinanziario'] == year]
    temp = temp.loc[temp['Indicatore_def_strutt'] == 1.0]
    for index, row in temp.iterrows():
        exact_matches += comuni.index[comuni['Comune'].str.lower() == 'comune di ' + str(row['denominazionecomune']).lower()].tolist()
    print(len(exact_matches))
    parametri_target[f'Target {year}'] = 0
    parametri_target.loc[exact_matches, f'Target {year}'] = 1

for year in np.arange(2016, 2019):
    exact_matches = []
    temp = parametri.loc[parametri['eserciziofinanziario'] == year]
    temp = temp.loc[temp['Indicatore_def_strutt'] == 1.0]
    for index, row in temp.iterrows():
        exact_matches += comuni.index[comuni['Comune'].str.lower() == 'comune di ' + str(row['descrizionecomune']).lower()].tolist()
    print(len(exact_matches))
    parametri_target[f'Target {year}'] = 0
    parametri_target.loc[exact_matches, f'Target {year}'] = 1

for year in np.arange(2019, 2025):
    exact_matches = []
    temp = parametri.loc[parametri['eserciziofinanziario'] == year]
    temp = temp.loc[temp['Indicatore_def_strutt'] == 1.0]
    for index, row in temp.iterrows():
        exact_matches += comuni.index[comuni['Comune'].str.lower() == str(row['denominazionesoggetto']).lower()].tolist()
    print(len(exact_matches))
    parametri_target[f'Target {year}'] = 0
    parametri_target.loc[exact_matches, f'Target {year}'] = 1

In [ ]:
for target in [1,2,3]:
    print(f'--- Target depth: {target} ---')
    print(f'{"Year":<6} {"F1 Score":<10} {"Precision":<10} {"Recall":<10}')
    for year in np.arange(2010, 2025-target):
        y = sum([comuni[f'Target {year + i + 1}'].values for i in range(target)])
        y = (y > 0).astype(int)
        f1 = f1_score(parametri_target[f'Target {year}'].values, y)
        p, r = precision_score(parametri_target[f'Target {year}'].values, y), recall_score(parametri_target[f'Target {year}'].values, y, zero_division=0)      
        print(f'{year:<6} {f1:<10.2f} {p:<10.2f} {r:<10.2f}')

In [ ]:
target = 1
r = 0
for year in np.arange(2009, 2025-target):
    y = sum([comuni[f'Target {year + i + 1}'].values for i in range(target)])
    y = (y > 0).astype(int)
    ratio = np.sum(y) / len(y)
    print(f'Year: {year}, Positive ratio: {ratio:.4f}')
    r += ratio
print(f'Average positive ratio: {r / 15:.4f}')

In [ ]:
s = 0
for year in range(2011, 2025):
    s += np.sum(comuni[f'Target {year}'].values)
print(f'Total positive cases from 2011 to 2024: {s}')

In [ ]:
acc = 0
p, r = 0, 0
cm = np.array([[0,0],[0,0]])
for year in np.arange(2010, 2024):
    y = comuni[f'Target {year + 1}'].values
    y = (y > 0).astype(int)
    acc += np.sum(y == parametri_target[f'Target {year}'].values)
    cmy = confusion_matrix(parametri_target[f'Target {year}'].values, y)
    cm += cmy
print(cm)
print(acc/(comuni.shape[0]*14))
print(p/14)
print(r/14)

TP, FP, FN, TN = cm[1,1], cm[0,1], cm[1,0], cm[0,0]

print(TP, FP, FN, TN)
precision = TP / (TP + FP)
recall = TP / (TP + FN)
accuracy = (TP + TN) / cm.sum()
f1 = 2 * (precision * recall) / (precision + recall)

print(f"precision: {precision:.3f}", f"recall: {recall:.3f}", f"f1: {f1:.3f}")

# Features Lists

In [ ]:
s = len(spese['2009'].columns) // 3
e = len(entrate['2009'].columns) // 3

features_names = [f'S{i+1} IMPEGNI' for i in range(s)] + [f'S{i+1} PAGAMENTI' for i in range(s)] + [f'S{i+1} RESIDUI' for i in range(s)]
features_names += [f'E{i+1} ACCERTAMENTI' for i in range(e)] + [f'E{i+1} RISCOSSIONI' for i in range(e)] + [f'E{i+1} RESIDUI' for i in range(e)]
# features_names += [f'I{i+1}' for i in range(10)]
# features_names += [f'A{i+1}' for i in range(5)]
# features_names += ['P'] + [f'Z{i+1}' for i in range(5)]

In [ ]:
table = []
for name, sigla, in zip(list(spese['2009'].columns) + list(entrate['2009'].columns), features_names):
    if 'CC ' in name:
        bef, af = name.split('CC ', maxsplit=1)
        af = af[0] + af[1:].lower()
        table.append((af, bef + 'CC ', sigla))
    elif 'CR ' in name:
        bef, af = name.split('CR ', maxsplit=1)
        af = af[0] + af[1:].lower()
        table.append((af, bef + 'CR ', sigla))
    else:
        bef, af = name.split(' ', maxsplit=1)
        af = af[0].upper() + af[1:].lower()
        table.append((af, bef, sigla))
table

In [ ]:
print("\n".join(" & ".join(map(str, row)) + r" \\" for row in table))